# 🛡️ Fake News Detection: 4-Step NLP & Machine Learning Pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)

### Option A (Academic Viva & Project Showcase)
This notebook implements the complete 4-step pipeline to classify news headlines and articles as **REAL (0)** or **FAKE (1)**:
1. **Data Preprocessing:** Lowercasing, punctuation & URL removal, NLTK stopwords elimination, and Porter Stemming.
2. **Feature Extraction:** TF-IDF (Term Frequency - Inverse Document Frequency) with unigrams & bigrams (5,000 features).
3. **Model Training:** Stratified 80/20 train/test split, Multinomial Naive Bayes, and Logistic Regression.
4. **Model Evaluation:** Accuracy (~93-96%), Precision, Recall, F1-Score, and Seaborn Confusion Matrix Heatmaps.

## 📦 Step 0: Install Dependencies & Download Dataset

In [ ]:
!pip install -q pandas numpy scikit-learn nltk matplotlib seaborn joblib

import urllib.request
import zipfile
import io
import os
import nltk
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Download NLTK corpora
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)

# Download George McIntire Fake and Real News Dataset
DATASET_URL = 'https://raw.githubusercontent.com/joolsa/fake_real_news_dataset/master/fake_or_real_news.csv.zip'
print('Downloading dataset...')
req = urllib.request.Request(DATASET_URL, headers={'User-Agent': 'Mozilla/5.0'})
with urllib.request.urlopen(req) as resp:
    with zipfile.ZipFile(io.BytesIO(resp.read())) as z:
        z.extractall('data')

df = pd.read_csv('data/fake_or_real_news.csv')
df['title'] = df['title'].fillna('').astype(str)
df['text'] = df['text'].fillna('').astype(str)
df['content'] = (df['title'] + ' ' + df['text']).str.strip()
df['label_num'] = (df['label'].str.upper() == 'FAKE').astype(int)
print(f'Dataset loaded successfully! Total records: {len(df)}')
print(df['label'].value_counts())

## 🧹 Step 1: Text Preprocessing
- Lowercasing
- URL and Punctuation removal
- Stopword removal using NLTK
- Stemming via PorterStemmer (`'running'` $\rightarrow$ `'run'`)

In [ ]:
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_text(text):
    # 1. Lowercase
    text = str(text).lower()
    # 2. Remove URLs & HTML tags
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'<.*?>', ' ', text)
    # 3. Remove punctuation and numbers
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    # 4. Tokenize, remove stopwords, and apply Porter Stemming
    tokens = text.split()
    stemmed = [stemmer.stem(w) for w in tokens if w not in stop_words and len(w) > 2]
    return ' '.join(stemmed)

print('Preprocessing text dataset (this may take ~45-60s)...')
df['clean_content'] = df['content'].apply(preprocess_text)
df = df[df['clean_content'].str.len() > 0].reset_index(drop=True)
print('Preprocessing complete! Sample preprocessed row:')
print(df[['title', 'clean_content', 'label']].head(2))

## 🔢 Step 2: Feature Extraction (TF-IDF)
Computers operate on numbers, not strings. We convert text into numerical vectors using **TF-IDF** (Term Frequency - Inverse Document Frequency).

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    strip_accents='unicode'
)

X = vectorizer.fit_transform(df['clean_content'])
y = df['label_num'].values
print(f'TF-IDF Feature Matrix Shape: {X.shape[0]} documents x {X.shape[1]} features')

## 🏋️ Step 3: Model Training (80/20 Train-Test Split)
Train **Multinomial Naive Bayes** and **Logistic Regression**.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f'Training Set: {X_train.shape[0]} samples | Testing Set: {X_test.shape[0]} samples')

# 1. Logistic Regression
lr_model = LogisticRegression(max_iter=1000, C=1.0, solver='liblinear', random_state=42)
lr_model.fit(X_train, y_train)

# 2. Multinomial Naive Bayes
nb_model = MultinomialNB(alpha=0.1)
nb_model.fit(X_train, y_train)

print('Both models successfully trained!')

## 📊 Step 4: Model Evaluation & Confusion Matrix
Evaluate Accuracy, Precision, Recall, F1-Score, and plot Confusion Matrix heatmaps.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

models = {'Logistic Regression': lr_model, 'Multinomial Naive Bayes': nb_model}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for idx, (name, clf) in enumerate(models.items()):
    y_pred = clf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    
    print(f'=== {name} ===')
    print(f'Accuracy:  {acc*100:.2f}% | Precision: {prec*100:.2f}% | Recall: {rec*100:.2f}% | F1: {f1*100:.2f}%')
    
    labels = np.array([
        [f'TN\n{cm[0,0]}', f'FP (Type I)\n{cm[0,1]}'],
        [f'FN (Type II)\n{cm[1,0]}', f'TP\n{cm[1,1]}']
    ])
    sns.heatmap(cm, annot=labels, fmt='', cmap='Blues', ax=axes[idx],
                xticklabels=['REAL (0)', 'FAKE (1)'], yticklabels=['REAL (0)', 'FAKE (1)'])
    axes[idx].set_title(f'{name}\nAccuracy: {acc*100:.2f}%', fontweight='bold')
    axes[idx].set_xlabel('Predicted Class')
    axes[idx].set_ylabel('Actual True Class')

plt.tight_layout()
plt.show()

## 🔮 Step 5: Interactive Custom News Prediction
Test any real or fake headline with live confidence probabilities and top keywords.

In [ ]:
def predict_news(news_text, model=lr_model):
    cleaned = preprocess_text(news_text)
    vec = vectorizer.transform([cleaned])
    pred = model.predict(vec)[0]
    probs = model.predict_proba(vec)[0]
    label = 'FAKE' if pred == 1 else 'REAL'
    
    print('----------------------------------------------------')
    print(f'Input Text: {news_text[:100]}...')
    print(f'PREDICTION:  ===> {label} (Class {pred}) <===')
    print(f'Probability Real: {probs[0]*100:.2f}% | Fake: {probs[1]*100:.2f}%')
    print(f'Confidence:       {probs[pred]*100:.2f}%')
    print('----------------------------------------------------')

# Test Real Sample
predict_news("NASA's James Webb Space Telescope observes oldest star clusters near cosmic dawn.")

# Test Fake Sample
predict_news("Secret banana peel concoction cures all cancers in 24 hours, Big Pharma threatens to delete video!")

## 💾 Step 6: Save Trained Models for Web App or Deployment

In [ ]:
import joblib
os.makedirs('models', exist_ok=True)
joblib.dump(vectorizer, 'models/tfidf_vectorizer.joblib')
joblib.dump(lr_model, 'models/logistic_regression.joblib')
joblib.dump(nb_model, 'models/multinomial_naive_bayes.joblib')
print('All model checkpoints saved to models/ directory!')